# Distributed Systems Project 1
## Final Google Colab Submission


## Team Members / Student IDs / Lab Number

- Student 1: [Name] - [Student ID]
- Student 2: [Name] - [Student ID]
- Lab Number: [Lab Number]


## Environment Setup

Install MRJob and create the output folder used in the submission.


In [1]:
!pip install mrjob
!mkdir -p outputs distutils


zsh:1: command not found: pip


Note: The following small compatibility files are included only to keep MRJob runnable on newer Colab Python runtimes.


In [2]:
%%writefile pipes.py
"""
Compatibility shim for the removed Python `pipes` module.

Python removed the `pipes` module in Python 3.13+.
Older libraries like `mrjob` still import `pipes.quote`.
This file restores that function using `shlex.quote`.
"""

from shlex import quote

__all__ = ["quote"]


Overwriting pipes.py


In [3]:
%%writefile distutils/__init__.py
"""Compatibility shim for the removed stdlib distutils package."""


Overwriting distutils/__init__.py


In [4]:
%%writefile distutils/spawn.py
"""Compatibility shim for distutils.spawn."""

from shutil import which


def find_executable(executable, path=None):
    """Return the path to an executable or None if not found."""
    return which(executable, path=path)


Overwriting distutils/spawn.py


In [5]:
%%writefile distutils/version.py
"""Compatibility shim for distutils.version."""

from packaging.version import Version


class LooseVersion:
    """Small compatibility wrapper around packaging.version.Version."""

    def __init__(self, version):
        self.vstring = str(version)
        self._version = Version(self.vstring)

    def __repr__(self):
        return f"LooseVersion ('{self.vstring}')"

    def __str__(self):
        return self.vstring

    def _coerce_other(self, other):
        if isinstance(other, LooseVersion):
            return other._version
        return Version(str(other))

    def __eq__(self, other):
        return self._version == self._coerce_other(other)

    def __lt__(self, other):
        return self._version < self._coerce_other(other)

    def __le__(self, other):
        return self._version <= self._coerce_other(other)

    def __gt__(self, other):
        return self._version > self._coerce_other(other)

    def __ge__(self, other):
        return self._version >= self._coerce_other(other)

    def __ne__(self, other):
        return self._version != self._coerce_other(other)


Overwriting distutils/version.py


## Exercise 1.1: Inverted Index


In [6]:
%%writefile task1.py
from __future__ import annotations

import os
import re
from collections.abc import Iterable
from typing import Any

from mrjob.compat import jobconf_from_env
from mrjob.job import MRJob


class MRInvertedIndex(MRJob):
    """Build an inverted index from multiple text documents."""

    JOBCONF = {"mapreduce.job.reduces": "1"}
    WORD_PATTERN = re.compile(r"\b[a-zA-Z0-9]+\b")

    @staticmethod
    def clean_line(line: str) -> str:
        return line.strip()

    @staticmethod
    def is_blank(line: str) -> bool:
        return not line.strip()

    @staticmethod
    def get_document_name() -> str:
        """Return the current input document name."""
        input_path = (
            jobconf_from_env("mapreduce.map.input.file")
            or jobconf_from_env("map.input.file")
            or ""
        )
        return os.path.basename(input_path)

    @classmethod
    def tokenize(cls, line: str) -> list[str]:
        """Normalize and tokenize a line into lowercase words."""
        return cls.WORD_PATTERN.findall(line.lower())

    def mapper(self, _: Any, line: str) -> Iterable[tuple[str, str]]:
        """Emit (word, document_name) for each word in the current document."""
        cleaned_line = self.clean_line(line)
        if self.is_blank(cleaned_line):
            return

        document_name = self.get_document_name()
        for word in self.tokenize(cleaned_line):
            yield word, document_name

    def reducer(self, key: str, values: Iterable[str]) -> Iterable[tuple[str, list[str]]]:
        """Emit each word once with a sorted unique list of document names."""
        unique_documents = sorted(set(values))
        yield key, unique_documents


if __name__ == "__main__":
    MRInvertedIndex.run()


Writing task1.py


In [7]:
%%writefile doc1.txt
big data systems use distributed computing to process massive datasets
mapreduce allows scalable processing across multiple machines
distributed systems improve reliability and fault tolerance
data engineers design pipelines for large scale analytics
parallel processing reduces computation time in big data environments
cloud platforms support distributed storage and computation
hadoop and spark are popular big data frameworks
data processing workflows include ingestion transformation and analysis
efficient systems require load balancing and resource management
modern analytics relies on scalable distributed architectures


Writing doc1.txt


In [8]:
%%writefile doc2.txt
mapreduce is widely used in big data analytics applications
distributed computing improves performance and scalability
data scientists analyze datasets using machine learning techniques
large scale systems require monitoring and optimization
cloud computing enables flexible infrastructure for analytics
parallel algorithms process data faster than sequential methods
data pipelines collect clean and transform raw information
analytics platforms provide visualization and reporting tools
fault tolerant systems recover automatically from failures
distributed environments support real time data processing


Writing doc2.txt


In [9]:
%%writefile doc3.txt
big data platforms rely on distributed storage systems
mapreduce enables parallel data processing across clusters
scalable architectures support growing datasets and users
machine learning models depend on high quality data
data engineering includes extraction loading and transformation
distributed databases ensure availability and consistency
real time analytics processes streaming data continuously
cloud services provide storage computation and networking
large systems must handle performance and security challenges
efficient data management improves decision making


Writing doc3.txt


In [10]:
!python task1.py doc1.txt doc2.txt doc3.txt 2>/dev/null | sort > task1_raw_output.txt

import ast
from pathlib import Path

raw_path = Path("task1_raw_output.txt")
final_path = Path("outputs/InvertedIndex_OUTPUT.txt")

with raw_path.open("r", encoding="utf-8") as infile, final_path.open("w", encoding="utf-8") as outfile:
    for line in infile:
        line = line.strip()
        if not line:
            continue

        word_raw, docs_raw = line.split("\t", 1)
        word = ast.literal_eval(word_raw)
        documents = ast.literal_eval(docs_raw)

        outfile.write(f"Word: {word}\n")
        outfile.write("Documents:\n")
        for document in documents:
            outfile.write(f"  - {document}\n")
        outfile.write("\n")

raw_path.unlink()

!cat outputs/InvertedIndex_OUTPUT.txt


## Exercise 1.2: Maximum Temperature


In [11]:
%%writefile task2.py
from __future__ import annotations

from collections.abc import Iterable
from typing import Any

from mrjob.job import MRJob


class MRMaxTemperature(MRJob):
    """Find the maximum temperature for each year."""

    @staticmethod
    def clean_line(line: str) -> str:
        return line.strip()

    @staticmethod
    def is_blank(line: str) -> bool:
        return not line.strip()

    @staticmethod
    def parse_line(line: str) -> tuple[str, float]:
        """Parse a line in the form YYYY-MM-DD,temperature."""
        date_str, temperature_str = line.split(",", 1)
        year = date_str[:4]
        temperature = float(temperature_str)
        return year, temperature

    def mapper(self, _: Any, line: str) -> Iterable[tuple[str, float]]:
        """Emit (year, temperature) for each valid reading."""
        cleaned_line = self.clean_line(line)
        if self.is_blank(cleaned_line):
            return

        year, temperature = self.parse_line(cleaned_line)
        yield year, temperature

    def reducer(self, key: str, values: Iterable[float]) -> Iterable[tuple[str, float]]:
        """Emit (year, max_temperature)."""
        yield key, max(values)


if __name__ == "__main__":
    MRMaxTemperature.run()


Writing task2.py


In [12]:
%%writefile temperature_readings.txt
2000-01-15,18.5
2000-03-10,25.0
2000-06-21,37.8
2000-08-05,35.4
2000-12-30,22.1
2001-02-11,19.2
2001-04-18,28.6
2001-07-25,39.1
2001-09-14,33.7
2001-11-20,24.3
2002-01-05,17.9
2002-03-22,26.4
2002-06-30,40.5
2002-08-19,38.2
2002-12-12,21.6
2003-02-17,20.0
2003-05-09,30.7
2003-07-27,41.3
2003-09-03,36.9
2003-12-28,23.4
2004-01-13,16.8
2004-04-02,27.9
2004-06-18,42.0
2004-08-11,39.5
2004-10-25,29.1
2005-02-07,18.3
2005-05-14,31.2
2005-07-22,43.6
2005-09-30,37.4
2005-12-19,22.8
2006-01-09,15.5
2006-03-28,29.0
2006-06-15,44.2
2006-08-23,40.7
2006-11-05,26.9
2007-02-16,17.1
2007-04-20,30.5
2007-07-10,45.0
2007-09-18,38.8
2007-12-27,24.6
2008-01-22,16.3
2008-03-30,28.4
2008-06-12,43.9
2008-08-29,41.5
2008-11-14,27.2
2009-02-03,18.9
2009-04-25,32.1
2009-07-19,46.3
2009-09-07,39.9
2009-12-21,25.4
2010-01-17,17.6
2010-03-12,29.8
2010-06-25,44.8
2010-08-04,42.3
2010-10-30,30.0


Writing temperature_readings.txt


In [13]:
!python task2.py temperature_readings.txt 2>/dev/null | sort > outputs/MaximumTemperature_OUTPUT.txt
!cat outputs/MaximumTemperature_OUTPUT.txt


## Exercise 1.3: Average Movie Rating


In [14]:
%%writefile task3.py
from __future__ import annotations

from collections.abc import Iterable
from typing import Any

from mrjob.job import MRJob


class MRAverageMovieRating(MRJob):
    """Compute the average rating for each movie."""

    @staticmethod
    def clean_line(line: str) -> str:
        return line.strip()

    @staticmethod
    def is_blank(line: str) -> bool:
        return not line.strip()

    @staticmethod
    def parse_line(line: str) -> tuple[str, float]:
        """Extract movie_id and rating from a CSV row."""
        _user_id, movie_id, rating = line.split(",", 2)
        return movie_id, float(rating)

    @staticmethod
    def compute_average(values: Iterable[float]) -> float:
        """Compute an average from an iterable of ratings using sum and count."""
        total = 0.0
        count = 0

        for value in values:
            total += value
            count += 1

        return round(total / count, 2)

    def mapper(self, _: Any, line: str) -> Iterable[tuple[str, float]]:
        """Emit (movie_id, rating) for each valid rating row."""
        cleaned_line = self.clean_line(line)

        if self.is_blank(cleaned_line):
            return

        if cleaned_line.startswith("user_id"):
            return

        movie_id, rating = self.parse_line(cleaned_line)
        yield movie_id, rating

    def reducer(self, key: str, values: Iterable[float]) -> Iterable[tuple[str, float]]:
        """Emit (movie_id, average_rating)."""
        yield key, self.compute_average(values)


if __name__ == "__main__":
    MRAverageMovieRating.run()


Writing task3.py


In [15]:
%%writefile movie_ratings.txt
user_id,movie_id,rating
1,101,4
2,101,5
3,101,3
4,102,2
5,102,3
6,102,4
7,103,5
8,103,4
9,103,5
10,104,3
11,104,2
12,104,4
13,105,5
14,105,4
15,105,5
16,101,4
17,102,3
18,103,4
19,104,3
20,105,5
21,106,2
22,106,3
23,106,4
24,107,5
25,107,4
26,107,5
27,108,3
28,108,2
29,108,4
30,109,5
31,109,4
32,109,5
33,110,3
34,110,4
35,110,2
36,101,5
37,102,4
38,103,5
39,104,3
40,105,4
41,106,3
42,107,5
43,108,3
44,109,4
45,110,3


Writing movie_ratings.txt


In [16]:
!python task3.py movie_ratings.txt 2>/dev/null | sort > outputs/AverageMovieRating_OUTPUT.txt
!cat outputs/AverageMovieRating_OUTPUT.txt


## Exercise 1.4: Product Sales Total


In [17]:
%%writefile task4.py
from __future__ import annotations

from collections.abc import Iterable
from typing import Any

from mrjob.job import MRJob


class MRProductSalesTotal(MRJob):
    """Compute total sales for each product."""

    @staticmethod
    def clean_line(line: str) -> str:
        return line.strip()

    @staticmethod
    def is_blank(line: str) -> bool:
        return not line.strip()

    @staticmethod
    def parse_line(line: str) -> tuple[str, float]:
        """Extract product_id and amount from a CSV row."""
        _transaction_id, product_id, amount = line.split(",", 2)
        return product_id, float(amount)

    @staticmethod
    def compute_total(values: Iterable[float]) -> float:
        """Sum all sales values."""
        total = 0.0
        for value in values:
            total += value
        return round(total, 2)

    def mapper(self, _: Any, line: str) -> Iterable[tuple[str, float]]:
        """Emit (product_id, amount) for each valid sales row."""
        cleaned_line = self.clean_line(line)

        if self.is_blank(cleaned_line):
            return

        if cleaned_line.startswith("transaction_id"):
            return

        product_id, amount = self.parse_line(cleaned_line)
        yield product_id, amount

    def reducer(self, key: str, values: Iterable[float]) -> Iterable[tuple[str, float]]:
        """Emit (product_id, total_sales)."""
        yield key, self.compute_total(values)


if __name__ == "__main__":
    MRProductSalesTotal.run()


Writing task4.py


In [18]:
%%writefile product_sales.txt
transaction_id,product_id,amount
1,P101,120.5
2,P102,75.0
3,P103,200.0
4,P101,50.0
5,P104,300.0
6,P102,25.5
7,P103,180.0
8,P105,90.0
9,P101,130.0
10,P104,220.0
11,P102,60.0
12,P103,95.5
13,P105,110.0
14,P106,250.0
15,P106,150.0
16,P101,80.0
17,P104,175.0
18,P102,40.0
19,P103,210.0
20,P105,70.0
21,P106,100.0
22,P107,300.0
23,P107,120.0
24,P108,55.0
25,P108,65.0
26,P109,400.0
27,P109,150.0
28,P110,90.0
29,P110,60.0
30,P101,95.0
31,P102,85.0
32,P103,140.0
33,P104,160.0
34,P105,130.0
35,P106,175.0
36,P107,200.0
37,P108,45.0
38,P109,220.0
39,P110,75.0
40,P101,60.0


Writing product_sales.txt


In [19]:
!python task4.py product_sales.txt 2>/dev/null | sort > outputs/ProductSalesTotal_OUTPUT.txt
!cat outputs/ProductSalesTotal_OUTPUT.txt


## Exercise 1.5: Most Frequent Word (Bonus)


In [20]:
%%writefile task5.py
from __future__ import annotations

import re
from collections.abc import Iterable, Iterator
from typing import Any

from mrjob.job import MRJob
from mrjob.step import MRStep


class MRMostFrequentWord(MRJob):
    """Find the most frequent word in a text document."""

    WORD_PATTERN = re.compile(r"\b[a-zA-Z0-9]+\b")

    @staticmethod
    def clean_line(line: str) -> str:
        return line.strip()

    @staticmethod
    def is_blank(line: str) -> bool:
        return not line.strip()

    def configure_args(self) -> None:
        """Add a flag to stop after the intermediate word-count step."""
        super().configure_args()
        self.add_passthru_arg(
            "--intermediate-only",
            action="store_true",
            help="Run only the word-frequency step and output intermediate counts.",
        )

    @classmethod
    def tokenize(cls, line: str) -> list[str]:
        """Normalize and tokenize a line into lowercase words."""
        return cls.WORD_PATTERN.findall(line.lower())

    def mapper(self, _: Any, line: str) -> Iterable[tuple[str, int]]:
        """Emit (word, 1) for each token."""
        cleaned_line = self.clean_line(line)
        if self.is_blank(cleaned_line):
            return

        for word in self.tokenize(cleaned_line):
            yield word, 1

    def reducer(self, key: str, values: Iterable[int]) -> Iterable[tuple[str, int]]:
        """Emit (word, total_count)."""
        yield key, sum(values)

    def mapper_find_max(self, word: str, frequency: int) -> Iterator[tuple[str, tuple[str, int]]]:
        """Send all word counts to one reducer for global max selection."""
        yield "most_frequent", (word, frequency)

    def reducer_find_max(
        self, _: str, values: Iterable[tuple[str, int]]
    ) -> Iterator[tuple[str, int]]:
        """Emit (most_frequent_word, frequency)."""
        most_frequent_word = ""
        max_frequency = -1

        for word, frequency in values:
            if frequency > max_frequency:
                most_frequent_word = word
                max_frequency = frequency
            elif frequency == max_frequency and word < most_frequent_word:
                most_frequent_word = word

        yield most_frequent_word, max_frequency

    def steps(self) -> list[MRStep]:
        """Run either the intermediate word-count step or the full two-step pipeline."""
        word_count_step = MRStep(mapper=self.mapper, reducer=self.reducer)

        if self.options.intermediate_only:
            return [word_count_step]

        return [
            word_count_step,
            MRStep(mapper=self.mapper_find_max, reducer=self.reducer_find_max),
        ]


if __name__ == "__main__":
    MRMostFrequentWord.run()


Writing task5.py


In [21]:
%%writefile bonus_text.txt
big data systems are transforming modern computing environments
data driven decisions help organizations improve performance
distributed systems allow processing of large datasets efficiently
mapreduce is a programming model used for large scale data processing
data analytics helps companies understand customer behavior
machine learning models depend on high quality data
big data platforms process structured and unstructured data
cloud computing enables scalable storage and computation
data engineers build reliable data pipelines
analytics systems support business intelligence applications
data processing requires efficient algorithms and scalable systems
parallel computing improves execution speed for large workloads
distributed storage ensures data availability and fault tolerance
modern applications generate massive amounts of data daily
data science combines statistics computing and domain knowledge
organizations rely on analytics to make informed decisions
efficient data management improves system performance
machine learning and analytics are core parts of big data systems
data visualization helps users understand complex information
large scale computing requires distributed architectures
big data technologies continue evolving rapidly
data processing frameworks simplify distributed computation
analytics platforms provide insights from raw data
data driven innovation creates competitive advantages
scalable systems handle increasing data volumes effectively
big data analytics supports predictive modeling
distributed computing enables faster data analysis
data platforms integrate storage processing and analytics tools
organizations invest heavily in data infrastructure
data quality directly impacts analytics results


Writing bonus_text.txt


In [22]:
!python task5.py --intermediate-only bonus_text.txt 2>/dev/null | sort > outputs/WordFrequency_OUTPUT.txt
!cat outputs/WordFrequency_OUTPUT.txt


In [23]:
!python task5.py bonus_text.txt 2>/dev/null > outputs/MostFrequentWord_OUTPUT.txt
!cat outputs/MostFrequentWord_OUTPUT.txt


## Final Results Summary

The following output files are available in the `outputs` folder.


In [24]:
from pathlib import Path

output_files = [
    ("Task 1 - Inverted Index", "outputs/InvertedIndex_OUTPUT.txt"),
    ("Task 2 - Maximum Temperature", "outputs/MaximumTemperature_OUTPUT.txt"),
    ("Task 3 - Average Movie Rating", "outputs/AverageMovieRating_OUTPUT.txt"),
    ("Task 4 - Product Sales Total", "outputs/ProductSalesTotal_OUTPUT.txt"),
    ("Task 5 - Word Frequency", "outputs/WordFrequency_OUTPUT.txt"),
    ("Task 5 - Most Frequent Word", "outputs/MostFrequentWord_OUTPUT.txt"),
]

for title, file_path in output_files:
    print(title)
    print("-" * len(title))
    print(Path(file_path).read_text(encoding="utf-8"))
    print()


Task 1 - Inverted Index
-----------------------


Task 2 - Maximum Temperature
----------------------------


Task 3 - Average Movie Rating
-----------------------------


Task 4 - Product Sales Total
----------------------------


Task 5 - Word Frequency
-----------------------


Task 5 - Most Frequent Word
---------------------------


